<a href="https://colab.research.google.com/github/shreyaganesh-123/CSA63--THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/EXPERIMENTS_08_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EXP 8

In [1]:
import hashlib

# Simulate a stolen password hash (a real system would never expose this!)
stolen_hash = hashlib.sha256("letmein".encode()).hexdigest()

wordlist = ["123456", "password", "admin", "letmein", "qwerty"]

print("Attempting dictionary attack on stolen hash...")

for word in wordlist:
    guess_hash = hashlib.sha256(word.encode()).hexdigest()

    if guess_hash == stolen_hash:
        print(f"Password cracked: '{word}'")
        break
    else:
        print("Password not found in wordlist.")

Attempting dictionary attack on stolen hash...
Password not found in wordlist.
Password not found in wordlist.
Password not found in wordlist.
Password cracked: 'letmein'


EXP 9

In [2]:
rules = [
    {"action": "ALLOW", "ip": "10.0.0.5", "port": 443},
    {"action": "ALLOW", "ip": "10.0.0.5", "port": 80},
    {"action": "DENY", "ip": "203.0.113.99", "port": None},  # block this IP entirely
    {"action": "DENY", "ip": None, "port": 23},  # block telnet from anyone
]

def check_packet(ip, port):
    for rule in rules:
        ip_match = rule["ip"] in (None, ip)
        port_match = rule["port"] in (None, port)

        if ip_match and port_match:
            return rule["action"]

    return "DENY"  # default: deny anything not explicitly allowed


packets = [
    ("10.0.0.5", 443),
    ("203.0.113.99", 80),
    ("10.0.0.9", 23),
    ("10.0.0.9", 8080),
]

for ip, port in packets:
    decision = check_packet(ip, port)
    print(f"Packet from {ip}:{port} -> {decision}")

Packet from 10.0.0.5:443 -> ALLOW
Packet from 203.0.113.99:80 -> DENY
Packet from 10.0.0.9:23 -> DENY
Packet from 10.0.0.9:8080 -> DENY


EXP 10

In [3]:
from collections import defaultdict

# Create the sample log so the script runs standalone (no upload needed)
with open("login_attempts.log", "w") as f:
    f.write("2026-07-08 09:00:01 LOGIN SUCCESS user=alice ip=10.0.0.5\n")
    f.write("2026-07-08 09:01:15 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:20 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:25 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:30 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:35 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:02:00 LOGIN SUCCESS user=bob ip=10.0.0.8\n")


def simple_ids(logfile, threshold=3):
    attempts = defaultdict(int)
    alerts = []

    with open(logfile) as f:
        for line in f:
            if "LOGIN FAILED" in line:
                ip = line.split("ip=")[1].strip()
                attempts[ip] += 1

                if attempts[ip] == threshold:
                    alerts.append(f"IDS ALERT: {threshold}+ failed logins from {ip}")

    return alerts


for alert in simple_ids("login_attempts.log", threshold=3):
    print(alert)

IDS ALERT: 3+ failed logins from 203.0.113.99


EXP 11

In [4]:
from cryptography.fernet import Fernet

# Install once: pip install cryptography

# Step 1: Generate a shared secret key (both sender & receiver need this)
key = Fernet.generate_key()
cipher = Fernet(key)

# Step 2: Sender encrypts the message before sending over the network
message = b"Transfer $500 to account 12345"
encrypted = cipher.encrypt(message)
print("Encrypted (what an eavesdropper sees):", encrypted)

# Step 3: Receiver decrypts it using the same key
decrypted = cipher.decrypt(encrypted)
print("Decrypted (what the receiver reads):", decrypted.decode())

Encrypted (what an eavesdropper sees): b'gAAAAABqZHT0fLuBOBZW3Dq9XnQMaqoWkoKrHAx35ERvKX4E58sek9Syrp-Tq2Wf36ApbX5h9pGrRTuR1mNlVShefFGNnlQW-MisMqYLy0Ztyg1flznUztg='
Decrypted (what the receiver reads): Transfer $500 to account 12345


EXP 12

In [5]:
import datetime

def generate_report(flagged_ips, source_log):
    lines = []
    lines.append("INCIDENT RESPONSE REPORT")
    lines.append(f"Generated: {datetime.datetime.now()}")
    lines.append(f"Source log: {source_log}")
    lines.append("-" * 40)

    if flagged_ips:
        lines.append(f"{len(flagged_ips)} suspicious IP(s) detected:")
        for ip, count in flagged_ips.items():
            lines.append(f" - {ip}: {count} failed login attempts")
        lines.append("Recommended action: Block listed IPs, force password reset.")
    else:
        lines.append("No suspicious activity detected.")

    return "\n".join(lines)


flagged = {"203.0.113.99": 5}
report = generate_report(flagged, "login_attempts.log")

print(report)

with open("incident_report.txt", "w") as f:
    f.write(report)

INCIDENT RESPONSE REPORT
Generated: 2026-07-25 08:34:35.503528
Source log: login_attempts.log
----------------------------------------
1 suspicious IP(s) detected:
 - 203.0.113.99: 5 failed login attempts
Recommended action: Block listed IPs, force password reset.


EXP 13

In [6]:
import datetime

def generate_report(flagged_ips, source_log):
    lines = []
    lines.append("INCIDENT RESPONSE REPORT")
    lines.append(f"Generated: {datetime.datetime.now()}")
    lines.append(f"Source log: {source_log}")
    lines.append("-" * 40)

    if flagged_ips:
        lines.append(f"{len(flagged_ips)} suspicious IP(s) detected:")
        for ip, count in flagged_ips.items():
            lines.append(f" - {ip}: {count} failed login attempts")
        lines.append("Recommended action: Block listed IPs, force password reset.")
    else:
        lines.append("No suspicious activity detected.")

    return "\n".join(lines)


flagged = {"203.0.113.99": 5}
report = generate_report(flagged, "login_attempts.log")

print(report)

with open("incident_report.txt", "w") as f:
    f.write(report)

INCIDENT RESPONSE REPORT
Generated: 2026-07-25 08:35:05.499226
Source log: login_attempts.log
----------------------------------------
1 suspicious IP(s) detected:
 - 203.0.113.99: 5 failed login attempts
Recommended action: Block listed IPs, force password reset.
